In [1]:
# scripts/run_workspace.py
import time
from pathlib import Path
from workspace import Workspace


ws = Workspace()
ws.components["core"].simulation(True)
#ws.components["core"].robot_api.jmove([0,0,0,0,0,0,0,0])


Switched to simulation API


In [ ]:
core = ws.components["core"]
tool = ws.components["microtube_gripper_1"]
J,C = core.IK(target_solid=tool.assembly["toolchanger_tool_side"], target_anchor="toolchanger_connection", target_offset=[0,0,-20,0,0,0], base_distance=400.0,
        rail_step=10.0, rail_span=2,tool_solid=core.toolchanger_robot_side, tool_anchor="toolchanger_connection", tool_offset=[0,0,0,0,0,0])

print(C)
if C == 2:
   core.robot_api.jmove(J, vel=200,accel=5000,jerk=50000)

# then we connect to the tool

# first we approach the toolchanger

J,C = core.IK(target_solid=tool.assembly["toolchanger_tool_side"], target_anchor="toolchanger_connection", target_offset=[0,0,0,0,0,0], base_distance=400.0,
        rail_step=10.0, rail_span=2,tool_solid=core.toolchanger_robot_side, tool_anchor="toolchanger_connection", tool_offset=[0,0,0,0,0,0])

if C == 2:
   core.robot_api.jmove(J, vel=200,accel=5000,jerk=50000)

# next we attach the tool
tool.assembly["toolchanger_tool_side"].attach_to(parent=core.toolchanger_robot_side, parent_anchor="toolchanger_connection", child_anchor="toolchanger_connection")   


J,C = core.IK(target_solid=core.robot_flange, target_anchor="output", target_offset=[0,0,-2,0,0,0], base_distance=400.0,
        rail_step=10.0, rail_span=2)

if C == 2:
   core.robot_api.jmove(J, vel=200,accel=5000,jerk=50000)

J,C = core.IK(target_solid=core.robot_flange, target_anchor="output", target_offset=[-40,0,0,0,0,0], base_distance=400.0,
        rail_step=10.0, rail_span=2)

if C == 2:
   core.robot_api.jmove(J, vel=200,accel=5000,jerk=50000)


J,C = core.IK(target_solid=core.robot_flange, target_anchor="output", target_offset=[0,0,-100,0,0,0], base_distance=400.0,
        rail_step=10.0, rail_span=2)

if C == 2:
   core.robot_api.jmove(J, vel=200,accel=5000,jerk=50000)   

SBS_adapter_source = ws.components["SBS_adapter_1"]
SBS_adapter_destination = ws.components["SBS_adapter_2"]


def pick_microtube(vial):
    tool_solid = tool.assembly["microtube_gripper"]
    microtube_source = None
    for child in SBS_adapter_source.assembly["SBS_adapter"].children["center"]:
      solid = child["child_solid"]
      component = ws.components[solid.component]
      if component.type == "microplate":
         microplate_source = component
         break

    if microplate_source is None:
        print("No microplate in source")
        return
    if microplate_source:
      microplate_solid = microplate_source.assembly["microplate"]
      for child in microplate_solid.children[vial]:
         solid = child["child_solid"]
         if(solid.type == "microtube"):
               microtube_source = solid
               break
    

    if microtube_source is None:
        print(f"No microtube at index {vial} in source microplate")
        return
    else:
        # we go on top of the vial from the microplate
        J,C = core.IK(target_solid=microplate_source.assembly["microplate"], target_anchor=vial, target_offset=[0,0,150,180.0,0,0], base_distance=360.0)
        if C == 2:
            core.robot_api.jmove(J, vel=2000,accel=2000,jerk=20000)
        else:
            print(f"Cannot reach above vial {vial} in source microplate")
            return
        # we go to the vial
        J,C = core.IK(target_solid=microtube_source, target_anchor="gripping_point", target_offset=[0,0,0,180.0,0,0], tool_solid=tool_solid, tool_anchor="gripping_point", base_distance=360.0)
        if C == 2:
            core.robot_api.jmove(J, vel=2000,accel=2000,jerk=20000)
        else:
            print(f"Cannot reach vial {vial} in source microplate")
            return

        # now we attach the microtube to the gripper
        microtube_source.attach_to(parent=tool_solid, parent_anchor="gripping_point", child_anchor="gripping_point", offset=[0,0,0,180,0,0])
        J,C = core.IK(target_solid=microplate_source.assembly["microplate"], target_anchor=vial, target_offset=[0,0,200,180.0,0,0], base_distance=360.0)
        if C == 2:
            core.robot_api.jmove(J, vel=2000,accel=2000,jerk=20000)
        else:
            print(f"Cannot reach above vial {vial} in source microplate")
            return
        




pick_microtube("A1")
print("Picked A1")


2
Picked A1
